# Day 24 — Lecture 6: How to fix Underfit & Overfit?

> Source: Apna College Prime AI/ML • Day 24 Lecture 6 • Video Duration: `06:26`
> Core Objective: Bias-Variance Trade-off samajhna aur Underfitting vs Overfitting fix karne ke 4 exact tarike seekhna.
> Key Parameters: `degree` (complexity), `alpha` (regularization), `n_features`, `n_samples`
> Dataset Used: Synthetic Curve Data (Trade-off Experiments)
> Core Concept: Complexity badhane par Bias ghatta hai aur Variance badhta hai — dono ke beech balanced middle point hi optimal model hai

## 🛠️ Step 0: Setup & Trade-off Experiment Data

Chalo ek synthetic noisy sine dataset banate hain jis par hum Bias-Variance Trade-off aur sabhi 4 fixes test kar sakein.

In [ ]:
# 1. Libraries import karo
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

# 2. Chhota noisy training set banao taaki Degree 9 clear overfit kare
np.random.seed(7)
X_train = np.linspace(0, 4, 12).reshape(-1, 1)
y_train = np.sin(X_train.ravel()) + np.random.normal(0, 0.35, 12)
X_test = np.linspace(0.1, 3.9, 30).reshape(-1, 1)
y_test = np.sin(X_test.ravel())
print("Samples -> Train:", len(X_train), "| Test:", len(X_test))

## ⚖️ Step 1: Bias-Variance Trade-off Curve

Dekho complexity badhane se **Bias** kam hota hai par **Variance** badhta hai, aur complexity ghatane se ulta hota hai.
- **Goal:** Aisi complexity chunna jahan Train error aur Test error dono balanced aur low hon.

In [ ]:
# 1. Degree 1 se 9 tak models fit karke Train vs Test MSE dekho
for deg in [1, 3, 6, 9]:
    poly = PolynomialFeatures(degree=deg)
    m = LinearRegression().fit(poly.fit_transform(X_train), y_train)
    tr_mse = mean_squared_error(y_train, m.predict(poly.transform(X_train)))
    te_mse = mean_squared_error(y_test, m.predict(poly.transform(X_test)))
    print(f"Degree {deg} -> Train MSE: {tr_mse:.4f} | Test MSE: {te_mse:.4f}")

## 📍 Step 2: Fix 1 & 3 — Complexity, Parameters & Features

| Lever | Underfitting Fix | Overfitting Fix |
|---|---|---|
| **Model Complexity** | Increase (e.g. Linear -> Poly) | Decrease (Simple model chuno) |
| **Parameters / Layers** | Increase parameters / NN layers | Decrease parameters / layers |
| **Number of Features** | Increase (Feature Engineering) | Decrease (Irrelevant features hatao) |

In [ ]:
# 1. Underfit (Deg 1) ko Complexity badhakar Deg 3 se fix karo
p1, p3 = PolynomialFeatures(1), PolynomialFeatures(3)
m1 = LinearRegression().fit(p1.fit_transform(X_train), y_train)
m3 = LinearRegression().fit(p3.fit_transform(X_train), y_train)

# 2. R2 improvement check karo
print("Underfit Deg 1 Test R2:", round(r2_score(y_test, m1.predict(p1.transform(X_test))), 4))
print("Fixed    Deg 3 Test R2:", round(r2_score(y_test, m3.predict(p3.transform(X_test))), 4))

## ⚙️ Step 3: Fix 2 — Regularization (`L1` / `L2`)

Lecture me bataya gaya:
- **Overfitting Fix:** Regularization strength **increase** karo (model weights par penalty lagti hai).
- **Underfitting Fix:** Regularization strength **decrease** (ya remove) karo.

In [ ]:
# 1. Overfit Degree 9 model par bina regularization vs Ridge regularization lagao
p9 = PolynomialFeatures(degree=9)
X_tr9, X_te9 = p9.fit_transform(X_train), p9.transform(X_test)

# 2. Ridge (L2) penalty se test error control karna dekho
ols9 = LinearRegression().fit(X_tr9, y_train)
ridge9 = Ridge(alpha=1.0).fit(X_tr9, y_train)
print("Deg 9 OLS   Test MSE:", round(mean_squared_error(y_test, ols9.predict(X_te9)), 4))
print("Deg 9 Ridge Test MSE:", round(mean_squared_error(y_test, ridge9.predict(X_te9)), 4))

## 🏆 Step 4: Fix 4 — Training Data Size Kab Kaam Aata Hai?

Ye golden interview rule yaad rakhna:
- **Underfitting:** Zyada training data add karne se **help NAHI hoti** (kyunki straight line abhi bhi curved pattern nahi pakad sakti).
- **Overfitting:** Zyada training data add karne se **bohot help hoti hai** (model variety dekhta hai aur variance ghatti hai).

In [ ]:
# 1. 150 samples ka bada dataset banao
np.random.seed(99)
X_big = np.linspace(0, 4, 150).reshape(-1, 1)
y_big = np.sin(X_big.ravel()) + np.random.normal(0, 0.15, 150)

# 2. Check karo ki zyada data par Underfit (Deg 1) vs Complex (Deg 9) kaise behave karta hai
m1_big = LinearRegression().fit(p1.fit_transform(X_big), y_big)
m9_big = LinearRegression().fit(p9.fit_transform(X_big), y_big)
print("More Data -> Deg 1 (Underfit) Test R2:", round(r2_score(y_test, m1_big.predict(p1.transform(X_test))), 4))
print("More Data -> Deg 9 (Overfit)  Test R2:", round(r2_score(y_test, m9_big.predict(p9.transform(X_test))), 4))